# Mobile Robots: From Kinematics to Autonomous Navigation

## Table of Contents

1. [Kinematic Models of Wheeled Robots](#1)
2. [Dubins Curves — Shortest Paths for Car-Like Robots](#2)
3. [Noisy Dynamics — When Models Meet Reality](#3)
4. [Bayesian State Estimation](#4)
5. [Kalman Filter and Extended Kalman Filter](#5)
6. [Particle Filter and Monte Carlo Localisation](#6)
7. [Line Following and PID Control](#7)
8. [Path Planning — Dijkstra, A\*, and RRT](#8)

### Notation

| Symbol | Meaning |
|--------|---------|
| $\mathbf{x} = (x, y, \theta)^\top$ | robot state — position and heading |
| $\mathbf{u}$ | control input (model-dependent) |
| $f(\mathbf{x}, \mathbf{u})$ | deterministic state-transition function |
| $\Delta t$ | time step |
| $v, \omega$ | linear / angular velocity |
| $\delta$ | front-wheel steering angle |
| $\kappa$ | signed path curvature |
| $\rho$ | minimum turning radius ($= 1/\kappa_{\max}$) |
| $\text{bel}(\mathbf{x})$ | belief — probability distribution over states |
| $\mathbf{Q}, \mathbf{R}$ | process / measurement noise covariance |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lib import (
    DubinsCar, DiffDrive, AckermannModel,
    DubinsCurves,
    VelocityMotionModel,
    RangeBearingModel, KalmanFilter, EKF,
    MCL,
    PIDController, PurePursuit,
    AStar, RRT,
)
from lib.viz import (
    plot_kinematic_comparison, plot_dubins_all_paths,
    plot_banana_vs_gaussian, plot_motion_cloud,
    plot_bayes_1d_steps,
    plot_kf_1d, plot_ekf_2d, plot_covariance_ellipse,
    plot_mcl_result, plot_particles,
    plot_pid_tuning,
    plot_planning_comparison, plot_rrt_tree,
)
from lib.simulator import _make_city_map

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)
%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True,
                      "grid.alpha": 0.2, "font.size": 10})

<a id="1"></a>
# 1. Kinematic Models of Wheeled Robots

All planar wheeled robots share the same **state space**
$\mathbf{x} = (x, y, \theta)^\top$ — position in the plane and heading
angle — but differ in how control inputs map to motion.

We derive equations of motion (EOM) for three fundamental models,
progressing from simplest to most practical.

---

## 1.1 The Dubins Car

The **simplest** model of a car-like vehicle:

* Moves at **constant** forward speed $v$.
* Steers via signed curvature $\kappa \in [-\kappa_{\max},\;\kappa_{\max}]$.
* Cannot reverse.

**Derivation.** &ensp;
In a small time interval $\Delta t$ the vehicle advances a distance
$s = v\,\Delta t$ along its heading.  The heading itself turns at rate
$\dot\theta = v\kappa$, because by definition the curvature of the
path is $\kappa = \mathrm{d}\theta / \mathrm{d}s$ and
$\mathrm{d}s/\mathrm{d}t = v$.

Projecting the velocity onto world-frame axes:

$$
\dot{\mathbf{x}}
= \begin{pmatrix} \dot x \\ \dot y \\ \dot\theta \end{pmatrix}
= \begin{pmatrix} v\cos\theta \\ v\sin\theta \\ v\,\kappa \end{pmatrix}
\tag{1.1}
$$

The **minimum turning radius** is
$\rho = 1/\kappa_{\max}$.
The robot can trace:

| Control | Path |
|---------|------|
| $\kappa = 0$ | straight line |
| $\kappa = +\kappa_{\max}$ | circle of radius $\rho$ turning **left** |
| $\kappa = -\kappa_{\max}$ | circle of radius $\rho$ turning **right** |

---

## 1.2 The Bicycle (Ackermann) Model

Generalise the Dubins car: allow **variable speed** $v$ and steer via
front-wheel angle $\delta$ instead of curvature directly.

**Derivation.** &ensp;
Consider a bicycle with wheelbase $L$ (distance between front and rear
axle).  When the front wheel is turned by angle $\delta$, the
instantaneous centre of rotation (ICR) lies on the line through the rear
axle.  By trigonometry, the **turning radius** measured from the ICR to
the rear axle is

$$
R = \frac{L}{\tan\delta}
\tag{1.2}
$$

The equivalent curvature is $\kappa = \tan(\delta)/L$,
so the angular rate is $\dot\theta = v\,\kappa = v\tan(\delta)/L$.
The EOM become

$$
\dot{\mathbf{x}}
= \begin{pmatrix} v\cos\theta \\ v\sin\theta \\
  \dfrac{v}{L}\tan\delta \end{pmatrix},
\qquad
\mathbf{u} = (v,\;\delta)
\tag{1.3}
$$

> **Connection to Dubins:** Setting $v = \text{const}$ and
> $\kappa = \tan(\delta)/L$ recovers the Dubins car (Eq. 1.1).

---

## 1.3 Differential Drive (Unicycle)

The most common platform for indoor mobile robots (Roomba, TurtleBot).
Two independently driven wheels of radius $r$ separated by distance $b$.

**Derivation.** &ensp;
Wheel velocities $v_R = r\omega_R$, $v_L = r\omega_L$.
The robot's body-frame linear and angular velocity are

$$
v = \frac{v_R + v_L}{2} = \frac{r(\omega_R + \omega_L)}{2},
\qquad
\omega = \frac{v_R - v_L}{b} = \frac{r(\omega_R - \omega_L)}{b}
\tag{1.4}
$$

Substituting into the heading-rate equation, the EOM in terms of
the **unicycle** controls $(v, \omega)$ are:

$$
\dot{\mathbf{x}}
= \begin{pmatrix} v\cos\theta \\ v\sin\theta \\ \omega \end{pmatrix}
\tag{1.5}
$$

> All three models share the structure
> $\dot{\mathbf{x}} = g(\mathbf{x})\,\mathbf{u}$ —
> they differ only in the mapping from control inputs to body-frame velocity.

In [ ]:
dt, T = 0.02, 8.0
steps = int(T / dt)

def simulate(model, controls, dt, steps):
    traj = [np.zeros(3)]
    for k in range(steps):
        traj.append(model.forward(traj[-1], *controls(k), dt))
    return np.array(traj)

dubins = DubinsCar(speed=1.0, kappa_max=0.5)
acker  = AckermannModel(wheelbase=2.0, max_steer=np.deg2rad(30))
dd     = DiffDrive(wheel_radius=0.05, wheel_base=0.3)

dubins_trajs = {
    r"$\kappa=0$ (straight)":       simulate(dubins, lambda k: (0.0,), dt, steps),
    r"$\kappa=+\kappa_{\max}$ (L)": simulate(dubins, lambda k: (0.5,), dt, steps),
    r"$\kappa=-\kappa_{\max}$ (R)": simulate(dubins, lambda k: (-0.5,), dt, steps),
}
acker_trajs = {
    r"$\delta=0°$":  simulate(acker, lambda k: (2.0, 0.0), dt, steps),
    r"$\delta=15°$": simulate(acker, lambda k: (2.0, np.deg2rad(15)), dt, steps),
    r"$\delta=30°$": simulate(acker, lambda k: (2.0, np.deg2rad(30)), dt, steps),
}
dd_trajs = {
    r"$v=1,\;\omega=0$":   simulate(dd, lambda k: (1.0, 0.0), dt, steps),
    r"$v=1,\;\omega=0.5$": simulate(dd, lambda k: (1.0, 0.5), dt, steps),
    r"$v=1,\;\omega=1.0$": simulate(dd, lambda k: (1.0, 1.0), dt, steps),
}
fig = plot_kinematic_comparison(
    {"dubins": dubins_trajs, "ackermann": acker_trajs, "diffdrive": dd_trajs},
    titles={"dubins": "Dubins Car (Eq. 1.1)",
            "ackermann": "Bicycle / Ackermann (Eq. 1.3)",
            "diffdrive": "Differential Drive (Eq. 1.5)"},
)
plt.show()

<a id="2"></a>
# 2. Dubins Curves — Shortest Paths for Car-Like Robots

Given two configurations
$\mathbf{q}_s = (x_s, y_s, \theta_s)$ and
$\mathbf{q}_g = (x_g, y_g, \theta_g)$,
what is the **shortest** path the Dubins car can follow?

### Key result (Dubins, 1957)

> The shortest bounded-curvature path without reversals
> consists of **at most three segments**, each being either
>
> * a **circular arc** of radius $\rho$ (turning **L**eft or **R**ight
>   at maximum curvature), or
> * a **S**traight line.

There are exactly **six** candidate path types:

| Family | Types | Mnemonic |
|--------|-------|----------|
| Circle–Straight–Circle | LSL, LSR, RSL, RSR | "drive between tangent circles" |
| Circle–Circle–Circle | LRL, RLR | "three-point turn" |

### Why only arcs and lines?

By Pontryagin's maximum principle, the optimal curvature control for
minimum path length is always *bang-bang* or zero:
$\kappa^*(t) \in \{-\kappa_{\max},\;0,\;+\kappa_{\max}\}$.
Hence every segment is a full-curvature arc or a straight line.

### Geometric construction (CSC example: LSL)

1. At the start, draw the left-turn circle $C_s^L$ (centre at distance
   $\rho$ to the **left** of heading).
2. At the goal, draw the left-turn circle $C_g^L$.
3. Find the **outer tangent** line between $C_s^L$ and $C_g^L$.
4. The path is: arc on $C_s^L$ → tangent → arc on $C_g^L$.

For each of the six types, we compute the arc and straight-segment
lengths analytically and pick the **shortest valid** path.

In [ ]:
solver = DubinsCurves(rho=2.0)
start = np.array([0.0, 0.0, 0.0])
goal  = np.array([10.0, 6.0, np.deg2rad(150)])

fig = plot_dubins_all_paths(solver, start, goal)
plt.show()

best = solver.shortest(start, goal)
print(f"Shortest path: {best.path_type}, length = {best.total_length:.2f}")

<a id="3"></a>
# 3. Noisy Dynamics — When Models Meet Reality

Real actuators are imprecise: motors have backlash, wheels slip, terrain
is uneven.  The deterministic model $\mathbf{x}_t = f(\mathbf{x}_{t-1},
\mathbf{u}_t)$ becomes **probabilistic**:

$$
\mathbf{x}_t = f(\mathbf{x}_{t-1},\;\mathbf{u}_t + \boldsymbol{\varepsilon}_t)
\tag{3.1}
$$

where $\boldsymbol{\varepsilon}_t$ is random noise in the executed control.

## Velocity Motion Model

For the unicycle (or diff-drive), noise enters through the commanded
velocity pair $(v, \omega)$.  Following Thrun et al., we model:

$$
\hat v = v + \varepsilon_v, \qquad
\hat\omega = \omega + \varepsilon_\omega
$$

with **control-dependent** variances:

$$
\varepsilon_v \sim \mathcal{N}\!\bigl(0,\;
  \alpha_1 v^2 + \alpha_2 \omega^2\bigr),
\qquad
\varepsilon_\omega \sim \mathcal{N}\!\bigl(0,\;
  \alpha_3 v^2 + \alpha_4 \omega^2\bigr)
\tag{3.2}
$$

The four parameters $\alpha_1 \ldots \alpha_4$ capture:

| Parameter | Physical meaning |
|-----------|-----------------|
| $\alpha_1$ | speed noise ∝ speed (tyre slip) |
| $\alpha_2$ | speed noise ∝ rotation (lateral wobble) |
| $\alpha_3$ | rotation noise ∝ speed |
| $\alpha_4$ | rotation noise ∝ rotation |

## The "Banana" Distribution

If we sample many realisations of $(\hat v, \hat\omega)$ and propagate
each through the deterministic kinematics, the resulting cloud of
next-states is **banana-shaped** — *not* Gaussian.

*Why?* Heading noise creates **angular** spread, while speed noise
creates **radial** spread.  When composed over a curved trajectory,
these produce a characteristic crescent.

A Gaussian (obtained by linearising the model) captures the mean
and covariance but misses this curvature — a limitation we will
revisit in §5–6.

In [ ]:
mm = VelocityMotionModel(alpha_1=0.1, alpha_2=0.01, alpha_3=0.01, alpha_4=0.1)
x0 = np.zeros(3)
v_cmd, omega_cmd, dt_cmd = 1.0, 0.3, 5.0

particles = mm.sample(x0, v_cmd, omega_cmd, dt_cmd, n=8000, rng=rng)

# Linearised Gaussian approximation
dd_model = DiffDrive()
G = dd_model.jacobian_state(x0, v_cmd, omega_cmd, dt_cmd)
V = dd_model.jacobian_noise(x0, v_cmd, omega_cmd, dt_cmd)
M = np.diag([mm.alpha_1 * v_cmd**2 + mm.alpha_2 * omega_cmd**2,
             mm.alpha_3 * v_cmd**2 + mm.alpha_4 * omega_cmd**2])
mu_lin = dd_model.forward(x0, v_cmd, omega_cmd, dt_cmd)
Sigma_lin = G @ np.eye(3) * 0.001 @ G.T + V @ M @ V.T

fig = plot_banana_vs_gaussian(particles, mu_lin, Sigma_lin)
plt.show()

<a id="4"></a>
# 4. Bayesian State Estimation

We have noisy motion *and* noisy sensors.
How do we estimate the robot's true state?

## 4.1 Bayes' Theorem — From First Principles

Start from the definition of conditional probability:

$$
P(A \mid B) = \frac{P(A \cap B)}{P(B)},
\qquad
P(B \mid A) = \frac{P(A \cap B)}{P(A)}
$$

Eliminate $P(A \cap B)$:

$$
\boxed{
P(A \mid B) = \frac{P(B \mid A)\; P(A)}{P(B)}
}
\tag{4.1}
$$

In our context $A = \mathbf{x}$ (state), $B = \mathbf{z}$ (measurement):

$$
\underbrace{p(\mathbf{x} \mid \mathbf{z})}_{\text{posterior}}
= \frac{
  \overbrace{p(\mathbf{z} \mid \mathbf{x})}^{\text{likelihood}}
  \;\;
  \overbrace{p(\mathbf{x})}^{\text{prior}}
}{
  \underbrace{p(\mathbf{z})}_{\text{normaliser}}
}
\tag{4.2}
$$

| Term | Robot interpretation |
|------|----------------------|
| prior $p(\mathbf{x})$ | what we believe **before** seeing $\mathbf{z}$ |
| likelihood $p(\mathbf{z}\mid\mathbf{x})$ | sensor model — how likely is $\mathbf{z}$ if state is $\mathbf{x}$? |
| posterior $p(\mathbf{x}\mid\mathbf{z})$ | updated belief **after** seeing $\mathbf{z}$ |
| normaliser $p(\mathbf{z})$ | ensures posterior integrates to 1 |

## 4.2 Recursive Bayes Filter

At each time step $t$ we receive a control $\mathbf{u}_t$ and a
measurement $\mathbf{z}_t$.  The belief is updated in two stages:

### Predict — propagate through motion model

$$
\overline{\text{bel}}(\mathbf{x}_t)
= \int p(\mathbf{x}_t \mid \mathbf{x}_{t-1}, \mathbf{u}_t)\;
       \text{bel}(\mathbf{x}_{t-1})\;
       d\mathbf{x}_{t-1}
\tag{4.3}
$$

This integral "smears" the belief according to motion uncertainty
(cf. the banana distribution).

### Update — incorporate measurement

$$
\text{bel}(\mathbf{x}_t)
= \eta\; p(\mathbf{z}_t \mid \mathbf{x}_t)\;
  \overline{\text{bel}}(\mathbf{x}_t)
\tag{4.4}
$$

where $\eta$ normalises the product to a valid distribution.

**Predict spreads, update sharpens.**  The following 1-D example makes
this concrete.

## 4.3 Example: Discrete Bayes Filter on a 1-D Grid

A robot lives on a line $x \in [0, 10]$, discretised into cells.
It moves **right** with Gaussian noise and gets noisy position
measurements.

In [ ]:
# 1-D discrete Bayes filter
n_cells = 50
xs = np.linspace(0, 10, n_cells)
dx = xs[1] - xs[0]

def make_prior(xs):
    bel = np.ones(len(xs))
    return bel / bel.sum()

def predict_1d(bel, shift_cells, sigma_cells):
    new = np.zeros_like(bel)
    for i in range(len(bel)):
        for j in range(len(bel)):
            d = (i - j - shift_cells)
            new[i] += bel[j] * np.exp(-0.5 * (d / sigma_cells)**2)
    return new / new.sum()

def update_1d(bel, measurement, sigma_cells, xs, dx):
    indices = (xs - xs[0]) / dx
    meas_idx = (measurement - xs[0]) / dx
    likelihood = np.exp(-0.5 * ((indices - meas_idx) / sigma_cells)**2)
    bel = bel * likelihood
    return bel / bel.sum()

bel = make_prior(xs)
true_pos = 2.0
move_sigma = 1.5   # cells
meas_sigma = 2.0   # cells
move_step  = 6.0   # cells

history = [("t=0  prior", bel.copy())]
true_positions = [true_pos]

# Step 1: predict (robot moves right)
bel = predict_1d(bel, move_step, move_sigma)
true_pos += move_step * dx
true_positions.append(true_pos)
history.append(("t=1  predict", bel.copy()))

# Step 1: update (measurement near true position)
meas = true_pos + rng.normal(0, meas_sigma * dx)
bel = update_1d(bel, meas, meas_sigma, xs, dx)
history.append(("t=1  update", bel.copy()))

# Step 2: predict
bel = predict_1d(bel, move_step, move_sigma)
true_pos += move_step * dx
true_positions.append(true_pos)
history.append(("t=2  predict", bel.copy()))

# Step 2: update
meas = true_pos + rng.normal(0, meas_sigma * dx)
bel = update_1d(bel, meas, meas_sigma, xs, dx)
history.append(("t=2  update", bel.copy()))

fig = plot_bayes_1d_steps(xs, history, true_positions, figsize=(16, 3))
fig.suptitle("1-D Discrete Bayes Filter:  predict spreads,  update sharpens",
             fontsize=11, y=1.04)
plt.show()

<a id="5"></a>
# 5. Kalman Filter and Extended Kalman Filter

The Bayes filter (§4) is **general** but the integral in the predict
step (Eq. 4.3) is usually intractable.
When both models are **linear** and all noise is **Gaussian**,
the integral has a beautiful closed-form solution: the **Kalman filter**.

## 5.1 Linear-Gaussian Setup

$$
\mathbf{x}_t = A\,\mathbf{x}_{t-1} + B\,\mathbf{u}_t + \mathbf{w}_t,
\qquad \mathbf{w}_t \sim \mathcal{N}(\mathbf{0}, Q)
$$
$$
\mathbf{z}_t = H\,\mathbf{x}_t + \mathbf{v}_t,
\qquad \mathbf{v}_t \sim \mathcal{N}(\mathbf{0}, R)
\tag{5.1}
$$

If $\text{bel}(\mathbf{x}_{t-1}) = \mathcal{N}(\boldsymbol{\mu}_{t-1},
\Sigma_{t-1})$, the belief stays Gaussian at every step.

## 5.2 Predict Step

A linear transform of a Gaussian is Gaussian:
if $\mathbf{x} \sim \mathcal{N}(\boldsymbol{\mu}, \Sigma)$
then $A\mathbf{x} + \mathbf{b} \sim
\mathcal{N}(A\boldsymbol{\mu} + \mathbf{b},\; A\Sigma A^\top)$.

Adding the independent process noise $Q$:

$$
\bar{\boldsymbol{\mu}}_t = A\,\boldsymbol{\mu}_{t-1} + B\,\mathbf{u}_t
\tag{5.2}
$$
$$
\bar\Sigma_t = A\,\Sigma_{t-1}\,A^\top + Q
\tag{5.3}
$$

## 5.3 Update Step

The measurement residual ("innovation") and its covariance:

$$
\mathbf{y}_t = \mathbf{z}_t - H\,\bar{\boldsymbol{\mu}}_t,
\qquad
S = H\,\bar\Sigma_t\,H^\top + R
\tag{5.4}
$$

The **Kalman gain** determines how to blend prediction with measurement:

$$
K = \bar\Sigma_t\,H^\top\,S^{-1}
\tag{5.5}
$$

$$
\boldsymbol{\mu}_t = \bar{\boldsymbol{\mu}}_t + K\,\mathbf{y}_t
\tag{5.6}
$$
$$
\Sigma_t = (I - K\,H)\,\bar\Sigma_t
\tag{5.7}
$$

### Intuition for the Kalman Gain

* **Sensor very precise** ($R$ small) $\;\Rightarrow\; K \to H^{-1}$:
  we trust the measurement.
* **Prediction very confident** ($\bar\Sigma$ small)
  $\;\Rightarrow\; K \to 0$: we trust the prediction.
* In between, $K$ interpolates optimally.

## 5.4 Example: 1-D Constant-Velocity Tracking

State $\mathbf{x} = (p, \dot p)^\top$ — position and velocity.
We observe noisy position measurements.

In [ ]:
# 1-D Kalman Filter: track position + velocity
dt_kf = 0.1
N_kf = 200
t_kf = np.arange(N_kf) * dt_kf

A = np.array([[1, dt_kf], [0, 1]])
B = np.zeros((2, 1))
H = np.array([[1.0, 0.0]])
Q = np.array([[0.01, 0], [0, 0.1]]) * dt_kf
R = np.array([[2.0]])

true_pos = np.cumsum(np.ones(N_kf) * 0.5 * dt_kf) + 0.02 * rng.standard_normal(N_kf).cumsum()
measurements = true_pos + rng.normal(0, np.sqrt(R[0, 0]), N_kf)

kf = KalmanFilter(A=A, B=B, H=H, Q=Q, R=R)
kf.reset(mu=np.array([0.0, 0.5]), Sigma=np.eye(2) * 5.0)

estimates, sigmas = [], []
for k in range(N_kf):
    kf.predict(np.zeros(1))
    kf.update(np.array([measurements[k]]))
    estimates.append(kf.mu.copy())
    sigmas.append(np.sqrt(kf.Sigma[0, 0]))

estimates = np.array(estimates)
sigmas = np.array(sigmas)

fig = plot_kf_1d(t_kf, true_pos, measurements, estimates[:, 0], sigmas,
                 ylabel="position $p$")
fig.axes[0].set_title("1-D Kalman Filter: position tracking")
plt.show()

## 5.5 Extended Kalman Filter (EKF)

The kinematic models of §1 are **nonlinear** — the Kalman filter
equations don't apply directly.  The EKF linearises
around the current estimate using first-order Taylor expansions.

Given nonlinear models

$$
\mathbf{x}_t = f(\mathbf{x}_{t-1}, \mathbf{u}_t) + \mathbf{w}_t,
\qquad
\mathbf{z}_t = h(\mathbf{x}_t) + \mathbf{v}_t
$$

compute Jacobians at the current mean:

$$
G_t = \left.\frac{\partial f}{\partial \mathbf{x}}\right|_{\boldsymbol{\mu}_{t-1}},
\qquad
V_t = \left.\frac{\partial f}{\partial \mathbf{u}}\right|_{\mathbf{u}_t},
\qquad
H_t = \left.\frac{\partial h}{\partial \mathbf{x}}\right|_{\bar{\boldsymbol{\mu}}_t}
\tag{5.8}
$$

Then apply the KF equations (5.2)–(5.7) with $A \to G_t$, $H \to H_t$,
and $Q \to V_t\,M\,V_t^\top$ where $M$ is the control noise covariance.

### EKF Localisation Demo

A diff-drive robot follows a circular path.  It observes
**range-bearing** measurements to known landmarks:

$$
h(\mathbf{x}_t, \mathbf{l}_j) =
\begin{pmatrix}
  \sqrt{(\ell_x - x)^2 + (\ell_y - y)^2} \\
  \text{atan2}(\ell_y - y,\; \ell_x - x) - \theta
\end{pmatrix}
\tag{5.9}
$$

The EKF fuses odometry prediction with landmark observations to
maintain a Gaussian belief over the robot pose.

In [ ]:
# EKF localisation on circular path with range-bearing landmarks
landmarks = np.array([[3, 3], [-3, 3], [-3, -3], [3, -3],
                      [0, 5], [5, 0], [-5, 0], [0, -5]], dtype=float)

dd_ekf = DiffDrive()
rb = RangeBearingModel(sigma_r=0.3, sigma_phi=np.deg2rad(5))
ekf = EKF(motion_model=dd_ekf, obs_model=rb,
          Q_coeffs=np.diag([0.02, 0.01]))
ekf.reset(mu=np.array([0.0, 0.0, 0.0]),
          Sigma=np.diag([0.5, 0.5, 0.1]))

v_ekf, omega_ekf, dt_ekf = 1.0, 0.3, 0.1
N_ekf = 300
true_traj, est_traj, Sigmas = [], [], []
state = np.zeros(3)

for k in range(N_ekf):
    state = dd_ekf.forward(state, v_ekf, omega_ekf, dt_ekf)
    true_traj.append(state.copy())
    ekf.predict(v_ekf, omega_ekf, dt_ekf)
    # observe landmarks within range 6m
    for lm in landmarks:
        d = np.hypot(lm[0] - state[0], lm[1] - state[1])
        if d < 6.0:
            z = rb.sample(state, lm, rng=rng)
            ekf.update(z, lm)
    est_traj.append(ekf.mu.copy())
    Sigmas.append(ekf.Sigma.copy())

true_traj = np.array(true_traj)
est_traj = np.array(est_traj)

fig = plot_ekf_2d(true_traj, est_traj, Sigmas, landmarks, every_n=10)
plt.show()

<a id="6"></a>
# 6. Particle Filter and Monte Carlo Localisation

The EKF maintains a single Gaussian — it fails when the posterior is
**multimodal** (e.g. the robot doesn't know which corridor it's in) or
**highly nonlinear**.

The particle filter represents the belief as a **set of weighted
samples** (particles), which can approximate *any* distribution.

## 6.1 From Importance Sampling to the Particle Filter

We want to estimate $p(\mathbf{x} \mid \mathbf{z})$ but can't evaluate
it directly.  **Importance sampling**: draw samples
$\mathbf{x}^{(i)}$ from a proposal distribution $q(\mathbf{x})$ and
re-weight:

$$
w^{(i)} = \frac{p(\mathbf{z} \mid \mathbf{x}^{(i)})\;
               p(\mathbf{x}^{(i)})}{q(\mathbf{x}^{(i)})}
\tag{6.1}
$$

If we choose the **motion model** as proposal
$q(\mathbf{x}_t) = p(\mathbf{x}_t \mid \mathbf{x}_{t-1}^{(i)},
\mathbf{u}_t)$, the prior cancels and:

$$
w_t^{(i)} = p(\mathbf{z}_t \mid \mathbf{x}_t^{(i)})
\tag{6.2}
$$

## 6.2 SIR Particle Filter Algorithm

At each time step:

1. **Predict**: For each particle $i$, sample
   $\mathbf{x}_t^{(i)} \sim
   p(\mathbf{x}_t \mid \mathbf{x}_{t-1}^{(i)}, \mathbf{u}_t)$
2. **Weight**: Set
   $w_t^{(i)} = p(\mathbf{z}_t \mid \mathbf{x}_t^{(i)})$,
   then normalise $\sum_i w^{(i)} = 1$.
3. **Resample**: Draw $N$ new particles with replacement,
   probability $\propto w^{(i)}$.

### Low-Variance Resampling

Multinomial resampling has high variance.
**Systematic** resampling draws one random number $r \sim U(0, 1/N)$
and picks particle $i$ when the cumulative weight passes
$r + (j-1)/N$.  This is $O(N)$ and much lower variance.

### Effective Sample Size

$$
N_{\text{eff}} = \frac{1}{\sum_{i=1}^N (w^{(i)})^2}
\tag{6.3}
$$

When $N_{\text{eff}}$ drops below a threshold (e.g. $N/2$), it signals
weight degeneracy — most weight is on very few particles — and
resampling is needed.

## 6.3 MCL Demo

We apply MCL to a diff-drive robot with range-bearing landmarks
(same scenario as §5, but with a particle filter instead of EKF).

In [ ]:
# MCL with range-bearing landmarks (simplified observation model)
mm_mcl = VelocityMotionModel(alpha_1=0.05, alpha_2=0.01, alpha_3=0.01, alpha_4=0.05)

class RangeBearingParticleObs:
    def __init__(self, landmarks, sigma_r=0.3, sigma_phi=np.deg2rad(5)):
        self.landmarks = landmarks
        self.sigma_r = sigma_r
        self.sigma_phi = sigma_phi

    def endpoints_from_pose(self, pose, ranges, angles):
        return np.column_stack([ranges, angles])

    def scan_log_prob(self, range_bearing_pairs):
        return 0.0  # not used directly

class SimpleMCL:
    def __init__(self, motion_model, landmarks, n_particles=500, sigma_r=0.3,
                 sigma_phi=np.deg2rad(5), rng=None):
        self.mm = motion_model
        self.landmarks = landmarks
        self.sigma_r = sigma_r
        self.sigma_phi = sigma_phi
        self.n = n_particles
        self.rng = rng or np.random.default_rng()
        self.particles = np.zeros((n_particles, 3))
        self.weights = np.ones(n_particles) / n_particles

    def init_gaussian(self, mu, Sigma):
        self.particles = self.rng.multivariate_normal(mu, Sigma, self.n)
        self.weights = np.ones(self.n) / self.n

    def predict(self, v, omega, dt):
        for i in range(self.n):
            self.particles[i] = self.mm.sample(
                self.particles[i], v, omega, dt, n=1, rng=self.rng)

    def update(self, true_pose, obs_range=6.0):
        log_w = np.zeros(self.n)
        rb_model = RangeBearingModel(sigma_r=self.sigma_r, sigma_phi=self.sigma_phi)
        for lm in self.landmarks:
            d_true = np.hypot(lm[0] - true_pose[0], lm[1] - true_pose[1])
            if d_true > obs_range:
                continue
            z = rb_model.sample(true_pose, lm, rng=self.rng)
            for i in range(self.n):
                log_w[i] += rb_model.log_prob(z, self.particles[i], lm)
        log_w -= log_w.max()
        self.weights = np.exp(log_w)
        self.weights /= self.weights.sum()

    def resample(self):
        positions = (self.rng.uniform() + np.arange(self.n)) / self.n
        cumsum = np.cumsum(self.weights)
        idx = np.searchsorted(cumsum, positions)
        idx = np.clip(idx, 0, self.n - 1)
        self.particles = self.particles[idx]
        self.weights = np.ones(self.n) / self.n

    def ess(self):
        return 1.0 / np.sum(self.weights**2)

    def mean_pose(self):
        x = np.sum(self.weights * self.particles[:, 0])
        y = np.sum(self.weights * self.particles[:, 1])
        s = np.sum(self.weights * np.sin(self.particles[:, 2]))
        c = np.sum(self.weights * np.cos(self.particles[:, 2]))
        return np.array([x, y, np.arctan2(s, c)])

mcl = SimpleMCL(mm_mcl, landmarks, n_particles=800,
                sigma_r=0.3, sigma_phi=np.deg2rad(5), rng=np.random.default_rng(7))
mcl.init_gaussian(np.array([0.0, 0.0, 0.0]), np.diag([1.0, 1.0, 0.3]))

state_mcl = np.zeros(3)
true_traj_mcl, est_traj_mcl, ess_hist = [], [], []

for k in range(N_ekf):
    state_mcl = dd_ekf.forward(state_mcl, v_ekf, omega_ekf, dt_ekf)
    true_traj_mcl.append(state_mcl.copy())
    mcl.predict(v_ekf, omega_ekf, dt_ekf)
    mcl.update(state_mcl, obs_range=6.0)
    ess_hist.append(mcl.ess())
    if mcl.ess() < mcl.n * 0.5:
        mcl.resample()
    est_traj_mcl.append(mcl.mean_pose())

true_traj_mcl = np.array(true_traj_mcl)
est_traj_mcl = np.array(est_traj_mcl)

fig = plot_mcl_result(true_traj_mcl, est_traj_mcl, landmarks,
                      mcl.particles, ess_hist)
plt.show()

<a id="7"></a>
# 7. Line Following and PID Control

Given a planned path, how does the robot **follow** it accurately?

## 7.1 Error Definition

For a diff-drive robot tracking a reference path, define:

* **Cross-track error** $e_d$: signed perpendicular distance from the
  robot to the nearest point on the path.
* **Heading error** $e_\theta$: angle between the robot's heading and the
  local path tangent.

## 7.2 PID Controller — Derivation

The PID control law produces an angular velocity command $\omega$ from
the heading error $e_\theta$:

$$
\omega(t) = \underbrace{K_p\;e_\theta(t)}_{\text{proportional}}
           + \underbrace{K_i \int_0^t e_\theta(\tau)\,d\tau}_{\text{integral}}
           + \underbrace{K_d\;\frac{de_\theta}{dt}}_{\text{derivative}}
\tag{7.1}
$$

| Term | Role | Too large → |
|------|------|-------------|
| **P** (proportional) | Corrects current error | Oscillation |
| **I** (integral) | Eliminates steady-state error | Overshoot, wind-up |
| **D** (derivative) | Dampens oscillation | Sluggish response, noise amplification |

Similarly, a linear velocity PID on distance-to-target controls speed.

### Tuning

The following demo shows how sweeping $K_p$ affects trajectory smoothness
and tracking error.  We keep $K_d$ fixed and set $K_i = 0$ (PD control).

## 7.3 Pure Pursuit — Geometric Alternative

Instead of PID, **pure pursuit** (Coulter 1992) computes a steering
command from a lookahead point on the path at distance $L_d$:

$$
\delta = \arctan\!\left(\frac{2\,L\,\sin\alpha}{L_d^2}\right)
\tag{7.2}
$$

where $\alpha$ is the angle to the lookahead point in the robot frame
and $L$ is the wheelbase.

Longer $L_d$ → smoother but slower convergence; shorter $L_d$ →
aggressive corrections.

In [ ]:
# PID tuning demo: diff-drive following a figure-8 path
dd_pid = DiffDrive()
dt_pid = 0.05
t_steps = 1000

# Reference path: figure-8
t_ref = np.linspace(0, 2 * np.pi, 200)
ref_path = np.column_stack([3 * np.sin(t_ref), 1.5 * np.sin(2 * t_ref), np.zeros(len(t_ref))])

gains = {
    r"$K_p=1$  (underdamped)": dict(kp_ang=1.0, kd_ang=0.1),
    r"$K_p=3$  (well-tuned)":  dict(kp_ang=3.0, kd_ang=0.3),
    r"$K_p=8$  (aggressive)":  dict(kp_ang=8.0, kd_ang=0.1),
}

results = {}
for label, g in gains.items():
    pid = PIDController(kp_lin=1.5, ki_lin=0.0, kd_lin=0.1,
                        kp_ang=g["kp_ang"], ki_ang=0.0, kd_ang=g["kd_ang"],
                        max_v=2.0, max_omega=3.0)
    pid.reset()
    state = np.array([0.0, 0.0, 0.0])
    traj, errors = [state.copy()], []
    wp_idx = 0
    for _ in range(t_steps):
        target = ref_path[wp_idx % len(ref_path)]
        if pid.reached(state, target, tol=0.2):
            wp_idx += 1
            target = ref_path[wp_idx % len(ref_path)]
        v, omega = pid.command(state, target, dt_pid)
        state = dd_pid.forward(state, v, omega, dt_pid)
        traj.append(state.copy())
        # cross-track error to nearest ref point
        dists = np.hypot(ref_path[:, 0] - state[0], ref_path[:, 1] - state[1])
        errors.append(dists.min())
    results[label] = (np.array(traj), np.array(errors))

fig = plot_pid_tuning(results)
fig.axes[0].plot(ref_path[:, 0], ref_path[:, 1], "k--", linewidth=1, alpha=0.4, label="reference")
fig.axes[0].legend(fontsize=7)
plt.show()

<a id="8"></a>
# 8. Path Planning — Dijkstra, A\*, and RRT

## 8.1 Problem Setup

Given an **occupancy grid** (map of obstacles), find a collision-free
path from start to goal.

**Graph abstraction**: Each free cell is a node; adjacent free cells are
connected by edges with cost equal to Euclidean distance (1 for
cardinal, $\sqrt{2}$ for diagonal neighbours).

## 8.2 Dijkstra's Algorithm

**Idea**: BFS explores nodes layer-by-layer (equal cost per step).
Dijkstra generalises to **weighted** edges by always expanding the
node with the **smallest** tentative distance.

**Algorithm**:

1. $g[\text{start}] = 0$, $\;g[\text{all others}] = \infty$.
2. Priority queue ordered by $g$.
3. While queue is not empty:
   - Pop node $n$ with smallest $g[n]$.
   - For each neighbour $m$:
     if $g[n] + \text{cost}(n, m) < g[m]$, update $g[m]$ and record parent.

**Optimality**: Dijkstra always processes the globally closest unvisited
node.  Since edge weights are non-negative, no later path to that node
can be shorter — so the greedy choice is optimal.

**Weakness**: Dijkstra explores **uniformly** in all directions —
it doesn't "know" where the goal is.

## 8.3 A\*: Adding a Heuristic

A\* augments Dijkstra with a **heuristic** $h(n)$ that estimates the
remaining distance from $n$ to the goal:

$$
f(n) = g(n) + h(n)
\tag{8.1}
$$

The priority queue is ordered by $f$ instead of $g$.

**Admissibility**: If $h(n) \le $ true distance for all $n$,
A\* finds the optimal path.  A common admissible heuristic on grids
is **Euclidean distance**.

> **A\* with $h = 0$ is exactly Dijkstra.**
> The heuristic focuses the search toward the goal,
> reducing the number of explored cells.

## 8.4 Rapidly-Exploring Random Trees (RRT)

Grid planners assume the robot can move in any direction.
For robots with **kinematic constraints** (e.g. a car that can't turn
in place), the grid path may be infeasible.

RRT builds a **tree** in configuration space:

1. Sample random configuration $\mathbf{x}_{\text{rand}}$.
2. Find **nearest** tree node $\mathbf{x}_{\text{near}}$.
3. **Steer** from $\mathbf{x}_{\text{near}}$ toward
   $\mathbf{x}_{\text{rand}}$ using the kinematic model for a few steps.
4. If the resulting segment is collision-free, add it to the tree.
5. Repeat until a node is within tolerance of the goal.

RRT is **probabilistically complete**: given enough iterations, it will
find a path if one exists — but the path is generally not optimal
(see RRT\* for asymptotic optimality).

In [ ]:
# Dijkstra vs A* comparison on a city-block map
occ, origin = _make_city_map(width=30, height=25, resolution=0.2,
                              rng=np.random.default_rng(42))
start_xy = np.array([3.0, 3.0])
goal_xy  = np.array([27.0, 22.0])

dijkstra = AStar(occ, resolution=0.2, origin=origin, use_heuristic=False)
astar    = AStar(occ, resolution=0.2, origin=origin, use_heuristic=True)

path_d, explored_d = dijkstra.plan(start_xy, goal_xy, record_explored=True)
path_a, explored_a = astar.plan(start_xy, goal_xy, record_explored=True)

fig = plot_planning_comparison(
    occ, 0.2, origin,
    {"Dijkstra ($h = 0$)": (path_d, explored_d),
     "A* (Euclidean $h$)":  (path_a, explored_a)},
)
plt.show()

In [ ]:
# RRT for a diff-drive robot
rrt = RRT(
    kinematic_model=DiffDrive(),
    occupancy=occ,
    resolution=0.2,
    origin=origin,
    dt=0.2,
    n_steps=5,
    max_iter=3000,
    goal_tol=1.5,
    goal_bias=0.15,
    rng=np.random.default_rng(42),
)
start_rrt = np.array([3.0, 3.0, 0.0])
goal_rrt  = np.array([27.0, 22.0, 0.0])
path_rrt = rrt.plan(start_rrt, goal_rrt)

fig = plot_rrt_tree(rrt._tree, path_rrt, occ, 0.2, origin)
plt.show()